In [ ]:
import pandas as pd
import numpy as np
import os
HOME_DIR = "/path/to/HOME_DIR"


In [ ]:
#!/usr/bin/env python3


import re
import pandas as pd


# --------------------------------------------------
# Paths
# --------------------------------------------------

HOME_DIR = "/path/to/HOME_DIR"

metadata_file = os.path.join(
    HOME_DIR,
    "data/downloads/wgEncodeAwgTfbsUniform/files.txt"
)

output_file = os.path.join(
    HOME_DIR,
    "data/label_name_with_TF_Celltype.csv"
)


# --------------------------------------------------
# Extract TF, cell type, and filename from files.txt
# --------------------------------------------------

rows = []

with open(metadata_file, "r") as f:
    for line in f:

        parts = line.strip().split("\t")

        if len(parts) < 2:
            continue

        filename = parts[0]
        metadata = parts[1]

        # Only use narrowPeak files
        if not filename.endswith(".narrowPeak.gz"):
            continue

        antibody_match = re.search(r"antibody=([^;]+)", metadata)
        cell_match = re.search(r"cell=([^;]+)", metadata)

        if antibody_match is None or cell_match is None:
            continue

        antibody = antibody_match.group(1).strip()
        cell = cell_match.group(1).strip()

        # Remove antibody annotations in parentheses
        tf = re.sub(r"_\([^)]+\)|\([^)]+\)", "", antibody)

        rows.append({
            "Transcription Factor": tf,
            "Cell Type": cell,
            "Filename": filename
        })


df = pd.DataFrame(rows)


# --------------------------------------------------
# Standardize TF names
# --------------------------------------------------

tf_name_mapping = {
    "GATA-2": "eGFP-GATA2",
    "GATA2": "eGFP-GATA2",
    "PAX5-N19": "PAX5-C20",
    "Pol2-4H8": "Pol2",
    "SIN3A": "Sin3Ak-20",
    "TCF7L2_C9B9": "TCF7L2",
    "UBTF": "UBF",
    "USF-1": "USF1",
    "eGFP-FOS": "c-Fos",
    "eGFP-JunD": "JunD",
}

df["Transcription Factor"] = (
    df["Transcription Factor"]
    .replace(tf_name_mapping)
)


# --------------------------------------------------
# Correct two historical filenames
# --------------------------------------------------

filename_mapping = {
    "wgEncodeAwgTfbsHaibGm12878Bclaf101388V0416101UniPk.narrowPeak.gz":
        "wgEncodeAwgTfbsHaibGm12878Bclaf101388V041610GTA1UniPk.narrowPeak.gz",

    "wgEncodeAwgTfbsSydhK562Bhlhe40nb100IggrabUniPk.narrowPeak.gz":
        "wgEncodeAwgTfbsSydhK562gataBhlhe40nb100IggrabUniPk.narrowPeak.gz",
}

df["Filename"] = df["Filename"].replace(filename_mapping)


# --------------------------------------------------
# Keep required columns
# --------------------------------------------------

df = df[
    [
        "Transcription Factor",
        "Cell Type",
        "Filename"
    ]
]


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

assert len(df) == 690, (
    f"Expected 690 experiments, found {len(df)}"
)

assert df["Filename"].nunique() == 690, (
    f"Expected 690 unique filenames, "
    f"found {df['Filename'].nunique()}"
)

assert df["Transcription Factor"].nunique() == 161, (
    f"Expected 161 unique TFs, "
    f"found {df['Transcription Factor'].nunique()}"
)

assert df["Transcription Factor"].notna().all()
assert df["Cell Type"].notna().all()
assert df["Filename"].notna().all()


# --------------------------------------------------
# Save
# --------------------------------------------------

df.to_csv(output_file, index=False)

print(f"Saved: {output_file}")
print(f"Experiments: {len(df)}")
print(f"Unique TFs: {df['Transcription Factor'].nunique()}")
print(f"Unique cell types: {df['Cell Type'].nunique()}")
print("All checks passed.")

In [ ]:
label_file_path = os.path.join(
    HOME_DIR,
    "data/label_name.txt"
)

mapping_file_path = os.path.join(
    HOME_DIR,
    "data/label_name_with_TF_Celltype.csv"
)

In [ ]:



# Load label file (has Feature_Index and Feature_Name)
df_labels = pd.read_csv(label_file_path, sep='\t')
print("=" * 80)
print("MERGING LABEL FILES")
print("=" * 80)

print("\n1. LOADED FILES:")
print("-" * 80)
print(f"Label file: {len(df_labels)} entries")
print(f"Columns: {list(df_labels.columns)}")
print(df_labels.head())

# Load mapping file (has TF, Cell Type, Filename, etc.)
df_mapping = pd.read_csv(mapping_file_path)
print(f"\nMapping file: {len(df_mapping)} entries")
print(f"Columns: {list(df_mapping.columns)}")
print(df_mapping.head())

# Extract base filename from Feature_Name for matching
# wgEncodeAwgTfbsBroadDnd41CtcfUniPk -> wgEncodeAwgTfbsBroadDnd41CtcfUniPk
df_labels['Feature_Name_Base'] = df_labels['Feature_Name'].str.replace('UniPk', '', regex=False)

# Extract base filename from Filename in mapping file for matching
# wgEncodeAwgTfbsBroadDnd41CtcfUniPk.narrowPeak.gz -> wgEncodeAwgTfbsBroadDnd41CtcfUniPk
df_mapping['Filename_Base'] = df_mapping['Filename'].str.replace('.narrowPeak.gz', '', regex=False)
df_mapping['Filename_Base'] = df_mapping['Filename_Base'].str.replace('UniPk', '', regex=False)

print("\n" + "=" * 80)
print("2. MATCHING FILENAMES:")
print("-" * 80)
print("Sample Feature_Name_Base from labels:")
print(df_labels['Feature_Name_Base'].head(10))
print("\nSample Filename_Base from mapping:")
print(df_mapping['Filename_Base'].head(10))

# Merge the two dataframes
# df_merged = df_labels.merge(
#     df_mapping[['Filename_Base', 'Transcription Factor', 'Cell Type', 'UniProt ID', 'TF_graph_file']], 
#     left_on='Feature_Name_Base', 
#     right_on='Filename_Base', 
#     how='left'
# )
df_merged = df_labels.merge(
    df_mapping[
        [
            'Filename_Base',
            'Transcription Factor',
            'Cell Type'
        ]
    ],
    left_on='Feature_Name_Base',
    right_on='Filename_Base',
    how='left',
    validate='one_to_one',
    sort=False
)

print("\n" + "=" * 80)
print("3. MERGE RESULTS:")
print("-" * 80)
print(f"Total entries after merge: {len(df_merged)}")
print(f"Successfully matched: {df_merged['Transcription Factor'].notna().sum()}")
print(f"Unmatched entries: {df_merged['Transcription Factor'].isna().sum()}")

if df_merged['Transcription Factor'].isna().sum() > 0:
    print("\nUnmatched entries (showing first 10):")
    unmatched = df_merged[df_merged['Transcription Factor'].isna()][['Feature_Index', 'Feature_Name']]
    print(unmatched.head(10))

# Create final dataframe with desired columns
df_final = df_merged[['Feature_Index', 'Feature_Name', 'Transcription Factor', 'Cell Type']].copy()
df_final.columns = ['Feature_Index', 'Feature_Name', 'TF', 'Cell_Type']
# Verify that the original label order was preserved
assert len(df_final) == 690, \
    f"Expected 690 experiments, got {len(df_final)}"

assert df_final["Feature_Index"].tolist() == list(range(690)), \
    "Feature_Index order changed during the merge"

assert df_final["TF"].notna().all(), \
    "Some TFs were not matched"

assert df_final["Cell_Type"].notna().all(), \
    "Some cell types were not matched"

assert df_final["TF"].nunique() == 161, \
    f"Expected 161 TFs, got {df_final['TF'].nunique()}"

print("Final mapping checks passed!")


print("\n" + "=" * 80)
print("4. FINAL MERGED FILE:")
print("-" * 80)
print(f"Shape: {df_final.shape}")
print(f"\nFirst 20 rows:")
print(df_final.head(20))

print("\n" + "=" * 80)
print("5. STATISTICS:")
print("-" * 80)
unique_tfs = df_final['TF'].dropna().unique()
unique_cells = df_final['Cell_Type'].dropna().unique()
print(f"Unique TFs: {len(unique_tfs)}")
print(f"Unique Cell Types: {len(unique_cells)}")
print(f"Total TF-Cell combinations: {len(df_final)}")

print(f"\nTop 20 most common TFs:")
print(df_final['TF'].value_counts().head(20))

print(f"\nTop 20 most common Cell Types:")
print(df_final['Cell_Type'].value_counts().head(20))

# Save the final merged file
output_path = os.path.join(
    HOME_DIR,
    "data/label_file_final_merged.csv"
)
df_final.to_csv(output_path, index=False)
print("\n" + "=" * 80)
print(f"Final merged file saved to: {output_path}")
print("=" * 80)

# Also save a TSV version if you prefer
output_tsv_path = output_path.replace('.csv', '.tsv')
df_final.to_csv(output_tsv_path, index=False, sep='\t')
print(f"TSV version saved to: {output_tsv_path}")
print("=" * 80)